# Step 7: Merge recovered gender into the analysis data

PKG placeholder codes (`UNK`/`INI`) are replaced by the gender inferred from the recovered forename (steps 5–6):

1. gender per AID by **majority over all papers** of the author; authors whose recovered names give as many F as M votes
   are set to `UNI` (conflict) and excluded from the analysis;
2. a gender already assigned in the PKG (F/M/UNI) is never overwritten;
3. authors without any recovered forename keep their PKG code.

**Inputs:** `data/derived/filtered_data.csv` (pure PKG, step 3), `data/derived/nan_ini_medline_names_and_gender.csv` (output of `step06_infer_gender.ipynb`)
**Output:** `data/derived/filtered_data_recovered.csv` (columns `AID, PMID, gender, begin_year, topics_num, PubYear`) – input for `step08_cox_dataprep.ipynb`.

In [ ]:
import duckdb

import os

BASE_DIR = os.path.abspath('..')                  # root of the replication package (notebooks run from code/)
DERIVED_DIR = f'{BASE_DIR}/data/derived'
PURE_CSV = f'{DERIVED_DIR}/filtered_data.csv'
GENDER_CSV = f'{DERIVED_DIR}/nan_ini_medline_names_and_gender.csv'
OUT_CSV = f'{DERIVED_DIR}/filtered_data_recovered.csv'

con = duckdb.connect()
con.execute("SET preserve_insertion_order = false")

## 1. Gender per AID from the recovered forenames (majority vote)

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE recovered_gender AS
WITH votes AS (
    SELECT CAST(AID AS BIGINT) AS aid,
           count(*) FILTER (WHERE gender = 'F') AS n_f,
           count(*) FILTER (WHERE gender = 'M') AS n_m,
           mode(gender) FILTER (WHERE gender IN ('UNI', 'INI', 'UNK')) AS other_mode
    FROM read_csv('{GENDER_CSV}', header = true, all_varchar = true)
    GROUP BY 1
)
SELECT aid, n_f, n_m,
       CASE WHEN n_f > n_m THEN 'F'
            WHEN n_m > n_f THEN 'M'
            WHEN n_f = n_m AND n_f > 0 THEN 'UNI'      -- conflicting votes
            ELSE other_mode END AS gender_recovered     -- NULL if nothing was recovered -> PKG code is kept below
FROM votes
""")
print(con.execute("""
SELECT gender_recovered, count(*) AS n_authors,
       avg(((n_f > 0) AND (n_m > 0))::INT) AS share_with_conflicting_votes
FROM recovered_gender GROUP BY 1 ORDER BY 2 DESC
""").df())

## 2. Replace placeholder codes in the pure PKG data and export

In [ ]:
con.execute(f"""
COPY (
    SELECT p.AID, p.PMID,
           CASE WHEN p.gender IN ('UNK', 'INI') OR p.gender IS NULL OR p.gender = ''
                THEN coalesce(r.gender_recovered, p.gender) ELSE p.gender END AS gender,
           p.begin_year, p.topics_num, p.PubYear
    FROM read_csv('{PURE_CSV}', header = true, all_varchar = true) p
    LEFT JOIN recovered_gender r ON r.aid = CAST(p.AID AS BIGINT)
) TO '{OUT_CSV}' (HEADER, DELIMITER ',')
""")
print('written', OUT_CSV)

In [ ]:
print(con.execute(f"""
SELECT
    gender,
    count(DISTINCT AID) AS n_authors,
    round(100.0 * count(DISTINCT AID) / sum(count(DISTINCT AID)) OVER (), 2) AS pct
FROM read_csv('{OUT_CSV}', header = true, all_varchar = true)
GROUP BY 1
ORDER BY 2 DESC
""").df())

In [ ]:
print(con.execute(f"""
SELECT
    gender,
    count(DISTINCT AID) AS n_authors,
    round(100.0 * count(DISTINCT AID) / sum(count(DISTINCT AID)) OVER (), 2) AS pct
FROM read_csv('{PURE_CSV}', header = true, all_varchar = true)
GROUP BY 1
ORDER BY 2 DESC
""").df())